# pipeline check

Every labelled page in `test/data/` goes through the `/detect` route (YOLO detection, then the CNN classifier), exactly as an API client would call it. The returned boxes are drawn over the original page so I can see what the pipeline misses. Predictions are matched to the labels the same way as the pytest report (greedy one-to-one, IoU >= 0.5).

| colour | meaning |
|---|---|
| green | detected, classified correctly |
| red | detected, classified wrong |
| grey | detected, labelled ambiguous (no right answer) |
| blue | missed: labelled but not detected |
| magenta | false positive: detected but not labelled |

The letter left of each box is the predicted state (`C` checked, `U` unchecked). For missed boxes it is the labelled state instead (`?` = ambiguous).

In [2]:
import os
import sys
from pathlib import Path

# Jupyter starts in notebooks/, run from backend/ instead so imports and paths resolve the way they do for the app and tests.
if Path.cwd().name == "notebooks":
    os.chdir("..")
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

from collections import Counter

from fastapi.testclient import TestClient
from IPython.display import display
from PIL import Image, ImageDraw, ImageFont

from main import app
from tests.evaluation import AMBIGUOUS, CHECKED, load_all_samples, match_pairs

client = TestClient(app)
samples = load_all_samples()

MAX_WIDTH = 1600  # fullres pages are 2550px wide, shrinking them keeps the notebook light
CATEGORIES = {
    "correct": (0, 160, 0),          # green
    "wrong": (220, 0, 0),            # red
    "ambiguous": (130, 130, 130),    # gray
    "missed": (0, 90, 255),          # blue
    "false positive": (220, 0, 220), # magenta
}

In [3]:
def run_pipeline(sample) -> list[dict]:
    """/detect boxes, with each bbox turned from API corners [x1, y1, x2, y2] into COCO [x, y, w, h] like the labels."""
    response = client.post("/detect", files={"image": (sample.path.name, sample.path.read_bytes())})
    response.raise_for_status()
    boxes = response.json()["boxes"]
    for box in boxes:
        x1, y1, x2, y2 = box["bbox"]
        box["bbox"] = [x1, y1, x2 - x1, y2 - y1]
    return boxes


def letter(is_checked: bool) -> str:
    return "C" if is_checked else "U"


def outcomes(sample, preds) -> list[tuple[list, str, str]]:
    """Every predicted and every missed box as (bbox, category, text)."""
    truths = sample.annotations
    pairs = dict(match_pairs([p["bbox"] for p in preds], [a.bbox for a in truths]))  # prediction -> truth
    marks = []
    for i, p in enumerate(preds):
        if i not in pairs:
            category = "false positive"
        elif truths[pairs[i]].label == AMBIGUOUS:
            category = "ambiguous"
        elif p["is_checked"] == (truths[pairs[i]].label == CHECKED):
            category = "correct"
        else:
            category = "wrong"
        marks.append((p["bbox"], category, letter(p["is_checked"])))
    claimed = set(pairs.values())
    for i, a in enumerate(truths):
        if i not in claimed:
            marks.append((a.bbox, "missed", "?" if a.label == AMBIGUOUS else letter(a.label == CHECKED)))
    return marks


def draw(sample, marks) -> Image.Image:
    page = Image.open(sample.path).convert("RGB")
    scale = min(1, MAX_WIDTH / page.width)
    page = page.resize((round(page.width * scale), round(page.height * scale)))  # resize first, so lines stay crisp
    canvas = ImageDraw.Draw(page)
    width = max(2, page.width // 500)
    font = ImageFont.load_default(size=max(12, page.width // 90))
    for (x, y, w, h), category, text in marks:
        x, y, w, h = (v * scale for v in (x, y, w, h))
        color = CATEGORIES[category]
        canvas.rectangle([x, y, x + w, y + h], outline=color, width=width)
        # Left of the box, on its own row: stacked rows of boxes leave no room above them.
        # A white outline instead of a filled background keeps the form text under the letter readable.
        canvas.text((x - width, y + h / 2), text, fill=color, font=font, anchor="rm", stroke_width=2, stroke_fill="white")
    return page

## Summary

In [4]:
results = {s.id: (s, outcomes(s, run_pipeline(s))) for s in samples}

print(f"{'sample':24} {'GT':>5} {'pred':>5} " + " ".join(f"{c:>14}" for c in CATEGORIES))
for sample_id, (sample, marks) in results.items():
    counts = Counter(category for _, category, _ in marks)
    n_pred = len(marks) - counts["missed"]
    print(f"{sample_id:24} {len(sample.annotations):5} {n_pred:5} " + " ".join(f"{counts[c]:14}" for c in CATEGORIES))

sample                      GT  pred        correct          wrong      ambiguous         missed false positive
fullres/canvas3.png         48    47             47              0              0              1              0
fullres/canvas1.png        118   117            113              2              0              3              2
fullres/canvas2.png         43    43             41              0              1              1              1
fullres/canvas4.png         79    81             79              0              0              0              2
lowres/canvas3.png          48    48             48              0              0              0              0
lowres/canvas1.png         118   116            115              0              0              3              1
lowres/canvas2.png          43    45             41              0              1              1              3
lowres/canvas4.png          79    82             79              0              0              0        

## Pages

In [ ]:
for sample_id, (sample, marks) in results.items():
    counts = Counter(category for _, category, _ in marks)
    print(f"{sample_id}: " + ", ".join(f"{counts[c]} {c}" for c in CATEGORIES))
    display(draw(sample, marks))